# 📝 텍스트 전처리·분석 과제 LV1 정답 — 정규표현식·형태소·불용어·빈도 (강사용)

각 문제의 **모범답안 + 해설(접근법·흔한 실수·대안)** 입니다. 학생이 스스로 풀어 본 뒤 비교하도록 안내하세요.

- 경로는 정답 노트북 기준 `../../day11_텍스트전처리_분석/data/`·`../../day11_텍스트전처리_분석/images/` 입니다.
- 그래프 문제(10·11)와 서술형(1)은 자가채점이 없습니다.
- 형태소 분석 결과는 kiwipiepy 기준으로 실측한 값입니다.

아래 셀을 먼저 실행해 이 단원 라이브러리와 한국어 형태소 분석기를 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한글 폰트를 준비합니다.
import re
import json
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from kiwipiepy import Kiwi
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

kiwi = Kiwi()   # 한국어 형태소 분석기(자바 불필요)

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `describe()` 로 별점 요약을, `describe(exclude='number')`로 범주형 텍스트 요약을 봅니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·요약
preview_reviews = pd.read_csv('../../day11_텍스트전처리_분석/data/reviews_band.csv')
print("행·열 크기:", preview_reviews.shape)
print("\n[앞 5행] head()"); display(preview_reviews.head())
print("\n[열·자료형·결측] info()"); preview_reviews.info()
print("\n[별점 요약] describe()"); display(preview_reviews.describe())
print("\n[범주형 요약] describe(exclude='number')"); display(preview_reviews.describe(exclude='number'))
print("\n[리뷰 원문 한 개]"); print(preview_reviews.loc[2, "text"])

## 1. 데이터 살펴보기 (서술형)
**배경**: 텍스트 분석의 첫걸음은 **원문을 눈으로 읽는 것**입니다. 위 `데이터 살펴보기` 셀의 출력을 보고, 이 자세교정밴드 리뷰 데이터에 대해 알게 된 사실을 정리해 보세요.

**요구사항**:
- 아래 서술 셀에 이 데이터에 대한 **관찰 2~3가지**를 문장으로 적으세요.
- 예를 들어: 리뷰가 몇 개인지, 별점(`rating`)의 분포·평균, 결측치가 있는지, 리뷰 본문(`text`)에 어떤 **노이즈**(특수문자·이모티콘·반복 자음 등)가 섞여 있는지 등 눈에 띄는 점을 적으면 됩니다.
- 정답은 하나가 아닙니다. 출력에서 실제로 확인되는 사실이면 됩니다.

> 이 문제는 자가채점(assert)이 없습니다. 아래 서술 셀에 직접 문장을 적고, 정답 노트북의 모범 서술과 비교해 보세요.

**모범 서술 (예시 — 정답은 여럿)**

- 리뷰는 모두 800개이고, 열은 `rating`(별점)과 `text`(리뷰 본문) 두 개다. 결측치는 없다.
- 별점 평균이 약 4.7로 **5점에 크게 치우쳐** 있다(800개 중 5점이 708개). 만족 리뷰가 대부분이다.
- 리뷰 본문은 구어체라 `ㅋㅋㅋ`·`^^`·`~~`·`!!!` 같은 **반복 자음·이모티콘·특수문자**가 섞여 있고, 띄어쓰기도 들쭉날쭉하다. 이런 노이즈를 정규표현식으로 걷어내야 단어 빈도를 제대로 셀 수 있다.

## 2. 정규표현식 — 특수문자·이모티콘 지우기
**배경**: 리뷰에는 `!!`·`^^`·`~~` 같은 기호가 잔뜩 섞여 있습니다. 단어를 세려면 **한글·영문·숫자·공백이 아닌 문자**를 전부 공백으로 바꿔야 합니다. 이럴 때 쓰는 도구가 **정규표현식**(`re` 모듈)입니다.

**요구사항**:
- 제공 셀의 `sample` 문자열 변수를 그대로 사용하세요.
- `re.sub` 와 패턴 `r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]'` 를 써서 **한글 음절·자모·영문·숫자·공백이 아닌 문자**를 모두 **공백 한 칸**(`' '`)으로 바꾼 결과를 `no_special` 변수에 담으세요.
  - `[^...]` 는 '대괄호 안의 것들이 **아닌** 문자'를 뜻합니다. `\s` 는 공백류입니다.
- 이 문제는 특수문자만 지웁니다. **공백이 여러 칸 남아도 정상**입니다(공백 정리는 문제 4에서).

**예시**
```
sample      →  자세가 확실히 좋아져요!! 어깨 펴지는 느낌ㅋㅋㅋㅋ 완전 강추~~ ^^
no_special  →  '자세가 확실히 좋아져요   어깨 펴지는 느낌ㅋㅋㅋㅋ 완전 강추     '
            →  !·^·~는 공백이 되고, '자세'·'어깨'와 한글 자모 'ㅋㅋㅋㅋ'는 남는다
```
<details><summary>힌트</summary>

```text
접근방법:
- 정규표현식의 부정 문자클래스로 '남길 문자'를 지정하고, 그 밖의 문자를 공백으로 치환한다.

세부구현:
1. re 모듈의 치환 함수에 패턴·바꿀 문자·대상 문자열을 순서대로 넘긴다
2. 패턴은 한글·영문·숫자·공백을 제외한 나머지를 가리키는 부정 문자클래스로 쓴다
3. 결과를 `no_special` 변수에 담아 출력해 기호가 사라졌는지 확인한다
```

</details>

In [ ]:
# [제공 코드] 이 문제에서 정제할 예시 문장입니다.
sample = '자세가 확실히 좋아져요!! 어깨 펴지는 느낌ㅋㅋㅋㅋ 완전 강추~~ ^^'
print(sample)

In [ ]:
no_special = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]', ' ', sample)
print(repr(no_special))

In [ ]:
# [자가채점]
assert isinstance(no_special, str)
assert no_special == '자세가 확실히 좋아져요   어깨 펴지는 느낌ㅋㅋㅋㅋ 완전 강추     '
for ch in '!^~':
    assert ch not in no_special
for word in ['자세', '어깨', '완전', '강추', 'ㅋㅋㅋㅋ']:
    assert word in no_special
print("✅ 문제2 통과!")

### 해설 — 문제 2
- **접근법**: `[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]` 는 '한글 음절·자모·영문·숫자·공백이 **아닌** 문자'입니다. 대괄호 안 맨 앞의 `^`가 '아닌'을 뜻해요(부정 문자클래스). 이걸 공백으로 치환하면 기호·이모지는 한 번에 사라지고, `ㄱ-ㅎㅏ-ㅣ` 범위에 넣은 `ㅋ`·`ㅎ` 같은 **한글 자모**는 다음 반복 축약 단계까지 남습니다.
- **흔한 실수**: 빈 문자열(`''`)로 치환하면 `좋아요!!어깨` 처럼 **붙어 있던 단어가 하나로 합쳐집니다**. 공백 한 칸으로 바꿔 단어 경계를 지키세요.
- **대안**: 지울 문자를 하나하나 나열(`replace`) 할 수도 있지만, 리뷰에 어떤 기호가 나올지 다 알 수 없으니 '남길 것만 정하고 나머지를 지우는' 방식이 훨씬 안전합니다.

## 3. 정규표현식 — 반복되는 글자 줄이기
**배경**: 구어체 리뷰에는 `ㅋㅋㅋㅋㅋ`·`좋아요!!!!`·`대박~~~~~` 처럼 **같은 글자가 여러 번** 반복됩니다. 이걸 그대로 두면 `ㅋㅋㅋ`과 `ㅋㅋㅋㅋ`가 서로 다른 단어로 취급됩니다. **3번 이상 반복되면 2번으로 줄여** 표기를 통일합시다.

**요구사항**:
- 제공 셀의 `sample3` 문자열 변수를 그대로 사용하세요.
- `re.sub` 와 패턴 `r'(.)\1{2,}'`, 치환문 `r'\1\1'` 을 써서 **같은 글자가 3번 이상 이어지면 2번으로** 줄인 결과를 `squeezed` 변수에 담으세요.
  - `(.)` 는 아무 글자 하나를 **그룹으로 기억**하고, `\1` 은 **그 기억한 글자와 같은 글자**를 뜻합니다(역참조). `{2,}` 는 '2번 이상 더'라는 수량자입니다.
  - 치환문의 `\1\1` 은 '기억한 글자를 두 번'이라는 뜻입니다.

**예시**
```
sample3   →  배송 빠르고ㅋㅋㅋㅋㅋ 착용감 좋아요!!!! 진짜 대박이에요~~~~~
squeezed  →  배송 빠르고ㅋㅋ 착용감 좋아요!! 진짜 대박이에요~~
```
<details><summary>힌트</summary>

```text
접근방법:
- 한 글자를 그룹으로 잡고, 그 글자가 두 번 넘게 더 이어지는 구간을 통째로 매치한다.
- 매치된 구간을 '그 글자 두 번'으로 바꾼다.

세부구현:
1. 패턴에서 아무 글자 하나를 소괄호로 묶어 그룹 1로 기억한다
2. 그 뒤에 역참조와 수량자를 붙여 같은 글자가 2번 이상 더 반복되는 부분을 잡는다
3. 치환문에는 역참조를 두 번 써서 두 글자만 남긴다
```

</details>

In [ ]:
# [제공 코드] 이 문제에서 정제할 예시 문장입니다.
sample3 = '배송 빠르고ㅋㅋㅋㅋㅋ 착용감 좋아요!!!! 진짜 대박이에요~~~~~'
print(sample3)

In [ ]:
squeezed = re.sub(r'(.)\1{2,}', r'\1\1', sample3)
print(squeezed)

In [ ]:
# [자가채점]
assert squeezed == '배송 빠르고ㅋㅋ 착용감 좋아요!! 진짜 대박이에요~~'
print("✅ 문제3 통과!")

### 해설 — 문제 3
- **접근법**: `(.)` 로 잡은 글자를 `\1` 로 다시 부르는 것이 **역참조**입니다. `\1{2,}` 는 '같은 글자가 2번 이상 더' 이므로, 패턴 전체는 '같은 글자 3번 이상'을 뜻합니다. 치환문 `r'\1\1'` 로 두 글자만 남기면 `ㅋㅋㅋㅋㅋ`→`ㅋㅋ`, `!!!!`→`!!` 가 됩니다.
- **흔한 실수**: 치환문을 `'\1\1'`(일반 문자열)로 쓰면 파이썬이 `\1` 을 이스케이프로 해석해 엉뚱한 결과가 나옵니다. 패턴과 치환문 **양쪽 모두 `r'...'`(raw 문자열)** 로 쓰세요.
- **대안**: `{2,}` 대신 `+`(1번 이상)를 쓰면 `좋아요!!` 처럼 **2번 반복도 줄여 버려** 한 글자만 남습니다. '3번 이상만 줄이기'가 목표라면 `{2,}` 가 맞습니다.

## 4. 정제 함수 완성 — clean_text
**배경**: 문제 2·3 의 정제에 **공백 정규화**(여러 칸 공백 → 한 칸, 양끝 공백 제거)를 더하면 재사용 가능한 **정제 함수**가 됩니다. 앞으로 모든 리뷰를 이 함수 하나로 처리합니다.

**요구사항**:
- 함수 `clean_text(text)` 를 정의하세요. 인자 `text` 를 받아 **정제된 문자열**을 **반환**합니다(출력 아님).
- 함수 안에서 다음 세 가지를 **순서대로** 적용하세요.
  1. 특수문자·이모티콘 제거 — 문제 2 의 패턴(한글·영문·숫자·공백이 아닌 문자 → 공백 한 칸)
  2. 반복 문자 축약 — 문제 3 의 패턴(같은 글자 3번 이상 → 2번)
  3. 공백 정규화 — 패턴 `r'\s+'` 로 **연속된 공백을 한 칸**으로 바꾸고, 마지막에 `strip()` 으로 양끝 공백을 없앤 뒤 반환
- 숫자·결측 같은 값이 들어와도 안전하도록 함수 첫 줄에서 `str(text)` 로 문자열로 바꿔 시작하세요.

**예시**
```
clean_text('어깨가 쫙 펴져요ㅎㅎㅎㅎ   완전 만족!!!   재구매 의사 있어요^^')
  →  '어깨가 쫙 펴져요ㅎㅎ 완전 만족 재구매 의사 있어요'
```
<details><summary>힌트</summary>

```text
접근방법:
- 문제 2·3 의 치환을 차례로 적용하고, 마지막에 공백을 한 칸으로 모은 뒤 양끝을 잘라 반환한다.

세부구현:
1. 인자를 문자열로 변환한다
2. 특수문자·이모티콘을 공백 한 칸으로 치환한다
3. 같은 글자 3번 이상 반복을 2번으로 줄인다
4. 연속 공백을 한 칸으로 줄이고, 양끝 공백을 제거한 결과를 반환한다
```

</details>

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]', ' ', text)   # 특수문자·이모티콘 제거
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)              # 반복 문자 축약
    text = re.sub(r'\s+', ' ', text)                        # 공백 정규화
    return text.strip()

print(clean_text('어깨가 쫙 펴져요ㅎㅎㅎㅎ   완전 만족!!!   재구매 의사 있어요^^'))

In [ ]:
# [자가채점]
assert clean_text('어깨가 쫙 펴져요ㅎㅎㅎㅎ   완전 만족!!!   재구매 의사 있어요^^') == '어깨가 쫙 펴져요ㅎㅎ 완전 만족 재구매 의사 있어요'
assert clean_text('  좋아요!!!  ') == '좋아요'
print("✅ 문제4 통과!")

### 해설 — 문제 4
- **접근법**: 세 치환의 **순서가 중요**합니다. 특수문자를 먼저 공백으로 바꾸면 공백이 잔뜩 생기는데, 마지막 공백 정규화가 그걸 한 칸으로 정리합니다. 순서를 바꿔 공백 정규화를 먼저 하면 뒤이어 생긴 공백이 그대로 남습니다.
- **흔한 실수**: `strip()` 을 빼면 문장 앞뒤에 공백이 남아, 나중에 공백으로 자를 때 빈 토큰이 생깁니다. 또 `return` 없이 `print` 만 하면 함수가 `None` 을 돌려줘 자가채점이 실패합니다.
- **대안**: 리뷰마다 이 함수를 부르는 대신 `reviews['text'].apply(clean_text)` 로 열 전체를 한 번에 정제할 수 있습니다(문제 8 에서 사용).

## 5. 형태소 분석 — 명사만 뽑기
**배경**: 한국어는 `자세가`·`자세를`·`자세는` 처럼 조사가 붙어, 공백으로 자르면 같은 단어가 다 다르게 셉니다. **형태소 분석기**(kiwipiepy)는 문장을 의미 단위로 쪼개고 **품사 태그**를 붙여 줍니다. 여기서는 **명사**만 골라 봅니다.

**요구사항**:
- `kiwi.tokenize(문장)` 은 형태소 객체의 리스트를 돌려줍니다. 각 객체는 `.form`(형태소 문자열)과 `.tag`(품사 태그)를 가집니다.
- 아래 예시 문장(`sample5`, 제공 셀에 있음)을 분석해, **품사 태그가 `NN` 으로 시작하는 형태소**(`NNG` 일반명사·`NNP` 고유명사 등)의 `.form` 만 모아 `nouns` 리스트에 담으세요.
  - 문자열의 `startswith` 로 태그가 `'NN'` 으로 시작하는지 확인할 수 있습니다.

**예시**
```
sample5  →  허리 교정에 좋은 밴드를 구매했어요
nouns    →  ['허리', '교정', '밴드', '구매']
```
<details><summary>힌트</summary>

```text
접근방법:
- 문장을 형태소 분석기에 넘겨 형태소 객체 목록을 얻는다.
- 각 객체의 품사 태그가 명사 계열로 시작하는지 확인해, 맞는 것만 형태소 문자열로 모은다.

세부구현:
1. kiwi 의 tokenize 에 문장을 넘긴다
2. 리스트 컴프리헨션으로 각 형태소를 순회한다
3. 태그가 명사 계열로 시작하는 것만 골라 form 값을 nouns 에 모은다
```

</details>

In [ ]:
# [제공 코드] 이 문제에서 분석할 예시 문장입니다. 형태소·품사 태그를 먼저 눈으로 확인해 보세요.
sample5 = '허리 교정에 좋은 밴드를 구매했어요'
for token in kiwi.tokenize(sample5):
    print(token.form, token.tag)

In [ ]:
nouns = [token.form for token in kiwi.tokenize(sample5) if token.tag.startswith('NN')]
print(nouns)

In [ ]:
# [자가채점]
assert nouns == ['허리', '교정', '밴드', '구매']
print("✅ 문제5 통과!")

### 해설 — 문제 5
- **접근법**: 위 제공 셀 출력을 보면 `허리 NNG` · `에 JKB` · `좋 VA` 처럼 형태소마다 태그가 붙습니다. 명사 태그는 `NNG`(일반명사)·`NNP`(고유명사)·`NNB`(의존명사) 등 모두 `NN` 으로 시작하므로, `startswith('NN')` 하나로 명사 계열을 전부 잡을 수 있습니다.
- **흔한 실수**: `token.tag == 'NN'` 으로 비교하면 아무것도 안 걸립니다 — 실제 태그는 `NNG`·`NNP` 처럼 더 깁니다. 또 `token` 자체를 담으면 문자열이 아니라 형태소 객체가 담기니 `.form` 을 꺼내세요.
- **대안**: 조사(`JKB`)·어미(`EF`)까지 다 남기면 단어 세기가 무의미해집니다. 명사만 남기는 것이 빈도 분석의 가장 단순한 출발점입니다.

## 6. 형태소 분석 — 명사·형용사·동사 중 2글자 이상
**배경**: 명사만 보면 `펴지`·`편하` 같은 **동작·상태 표현**을 놓칩니다. 리뷰 분석에서는 **명사(NN…)·형용사(VA…)·동사(VV…)** 를 함께 봅니다. 또 한 글자짜리 형태소는 뜻이 모호해 보통 버립니다.

**요구사항**:
- 아래 예시 문장(`sample6`, 제공 셀에 있음)을 형태소 분석해, 다음 **두 조건을 모두** 만족하는 형태소의 `.form` 만 `tokens` 리스트에 담으세요.
  1. 품사 태그가 `NN`, `VA`, `VV` 중 하나로 시작한다 (`startswith` 에 세 태그의 **튜플**을 넘기면 한 번에 확인됩니다)
  2. 형태소의 길이가 **2글자 이상**이다

**예시**
```
sample6  →  어깨가 펴지고 자세가 좋아지는 느낌이 들어요
tokens   →  ['어깨', '펴지', '자세', '느낌']
           ('좋' 은 형용사지만 1글자라 빠지고, '펴지'(동사)·'느낌'(명사)은 남는다)
```
<details><summary>힌트</summary>

```text
접근방법:
- 문제 5 와 같은 방식이되, 태그 조건을 명사·형용사·동사 세 가지로 넓히고 길이 조건을 추가한다.

세부구현:
1. kiwi 의 tokenize 에 문장을 넘긴다
2. 각 형태소의 태그가 세 태그 중 하나로 시작하는지 확인한다(startswith 는 튜플을 받는다)
3. 형태소 문자열의 길이가 2 이상인지 함께 확인해 tokens 에 모은다
```

</details>

In [ ]:
# [제공 코드] 이 문제에서 분석할 예시 문장입니다. 형태소·품사 태그를 먼저 눈으로 확인해 보세요.
sample6 = '어깨가 펴지고 자세가 좋아지는 느낌이 들어요'
for token in kiwi.tokenize(sample6):
    print(token.form, token.tag)

In [ ]:
tokens = [token.form for token in kiwi.tokenize(sample6)
          if token.tag.startswith(('NN', 'VA', 'VV')) and len(token.form) > 1]
print(tokens)

In [ ]:
# [자가채점]
assert tokens == ['어깨', '펴지', '자세', '느낌']
print("✅ 문제6 통과!")

### 해설 — 문제 6
- **접근법**: `startswith` 는 **튜플**을 받으면 '이 중 하나로 시작하면 참'이 됩니다. 그래서 `('NN', 'VA', 'VV')` 한 번으로 명사·형용사·동사를 모두 잡습니다. 길이 조건 `len(...) > 1` 은 `좋`·`들` 같은 1글자 형태소를 걸러 줍니다.
- **흔한 실수**: 조건 두 개를 `or` 로 이으면 1글자 명사가 그대로 남습니다. **두 조건은 `and`** 로 이어야 합니다.
- **대안**: 태그 조건을 문자열 세 번(`or` 3개)으로 써도 되지만 튜플 한 번이 훨씬 짧고 안전합니다. 이 조합(명사·형용사·동사 + 2글자 이상)이 앞으로 쓸 **표준 토큰화 규칙**입니다.

## 7. 불용어 제거 — 일반 불용어 목록 적용
**배경**: 형태소를 뽑아도 `것`·`시간`·`정도` 처럼 **어느 문서에나 나오는 흔한 말**이 섞입니다. 이런 단어를 **불용어**(stopword)라 부르고, 미리 만들어진 목록으로 걸러 냅니다. `data/stopwords_ko.json` 에 한국어 일반 불용어가 들어 있습니다.

**요구사항**:
- `json.load` 로 `data/stopwords_ko.json` 을 읽어 **집합(set)** 으로 만들어 `stopwords` 변수에 담으세요(파일 내용은 단어 리스트입니다). 집합의 크기는 **679** 입니다.
  - 집합으로 만들면 `단어 in stopwords` 확인이 빠릅니다.
- 아래 예시 문장(`sample7`, 제공 셀에 있음)을 문제 6 의 규칙(명사·형용사·동사 + 2글자 이상)으로 토큰화한 결과를 `tokens_before` 리스트에 담으세요.
- `tokens_before` 에서 `stopwords` 에 들어 있는 단어를 제거한 결과를 `tokens_after` 리스트에 담으세요.

**예시**
```
len(stopwords)  →  679
sample7         →  처음 착용할 때는 조금 불편했지만 시간 지나니 어깨가 편해요
tokens_before   →  ['처음', '착용', '불편', '시간', '지나', '어깨', '편하']   (7개)
tokens_after    →  ['처음', '착용', '불편', '지나', '어깨', '편하']   (6개 — 불용어 '시간' 제거)
```
<details><summary>힌트</summary>

```text
접근방법:
- 불용어 파일을 읽어 집합으로 만든 뒤, 토큰 리스트에서 그 집합에 속한 단어를 걸러 낸다.

세부구현:
1. 파일을 열어 json 으로 읽고 집합으로 변환해 `stopwords` 변수에 담는다
2. 문제 6 의 규칙으로 예시 문장을 토큰화해 `tokens_before` 변수에 담는다
3. tokens_before 를 순회하며 불용어 집합에 없는 단어만 tokens_after 에 모은다
```

</details>

In [ ]:
# [제공 코드] 이 문제에서 사용할 예시 문장입니다.
sample7 = '처음 착용할 때는 조금 불편했지만 시간 지나니 어깨가 편해요'
print(sample7)

In [ ]:
with open('../../day11_텍스트전처리_분석/data/stopwords_ko.json', encoding='utf-8') as f:
    stopwords = set(json.load(f))
print("불용어 개수:", len(stopwords))

tokens_before = [token.form for token in kiwi.tokenize(sample7)
                 if token.tag.startswith(('NN', 'VA', 'VV')) and len(token.form) > 1]
tokens_after = [word for word in tokens_before if word not in stopwords]
print("제거 전:", tokens_before)
print("제거 후:", tokens_after)

In [ ]:
# [자가채점]
assert len(stopwords) == 679
assert tokens_before == ['처음', '착용', '불편', '시간', '지나', '어깨', '편하']
assert tokens_after == ['처음', '착용', '불편', '지나', '어깨', '편하']
print("✅ 문제7 통과!")

### 해설 — 문제 7
- **접근법**: 불용어는 **집합(set)** 으로 두는 것이 정석입니다. 리스트로 두면 `in` 확인이 원소 수에 비례해 느려지지만, 집합은 거의 즉시 확인됩니다(토큰 수만 개를 걸러야 하니 차이가 큽니다).
- **흔한 실수**: 불용어를 **토큰화 전 원문**에서 지우려 하면 `시간이`·`시간을` 같은 형태를 못 잡습니다. **형태소로 쪼갠 뒤** 걸러야 합니다.
- **대안**: 이 일반 목록만으로는 부족합니다 — `제품`·`구매`처럼 **이 도메인에서만 의미 없는 단어**가 그대로 남거든요. 그걸 문제 9 에서 직접 처리합니다.

## 8. 단어 빈도 — 리뷰 800개에서 가장 많이 나온 단어
**배경**: 정제·토큰화·불용어 제거를 **전체 리뷰**에 적용하면 드디어 단어를 셀 수 있습니다. `collections.Counter` 로 빈도를 세고 상위 단어를 봅니다.

**요구사항**:
- `data/reviews_band.csv` 를 불러와 `reviews` 변수에 담으세요.
- 리뷰 800개의 `text` 를 각각 **문제 4 의 `clean_text` 로 정제 → 문제 6 의 규칙으로 토큰화 → 문제 7 의 `stopwords` 제거** 한 뒤, 모든 리뷰의 토큰을 **하나의 리스트** `all_tokens` 에 모으세요.
- `Counter` 로 `all_tokens` 의 빈도를 세어 `counter_general` 변수에 담으세요.
- `counter_general.most_common(20)` 으로 상위 20개를 출력해 **눈으로 확인**하세요 — 다음 문제에서 이 목록을 보고 판단해야 합니다.

**예시**
```
len(all_tokens)                    →  17212
counter_general.most_common(3)     →  [('자세', 1156), ('착용', 940), ('어깨', 893)]
counter_general['교정']             →  577
```
<details><summary>힌트</summary>

```text
접근방법:
- 리뷰마다 정제 → 토큰화 → 불용어 제거를 거친 토큰들을 하나의 리스트에 누적한다.
- 그 리스트를 Counter 에 넣어 단어별 등장 횟수를 얻는다.

세부구현:
1. csv 를 읽어 `reviews` 변수에 담는다
2. 빈 리스트 all_tokens 를 만들고, 리뷰 본문을 하나씩 순회한다
   2-1. clean_text 로 정제한다
   2-2. 형태소 분석 후 명사·형용사·동사이면서 2글자 이상인 것만 고른다
   2-3. 불용어 집합에 없는 단어만 all_tokens 에 더한다(리스트 확장)
3. Counter 에 all_tokens 를 넣어 counter_general 을 만들고 상위 20개를 출력한다
```

</details>

In [ ]:
reviews = pd.read_csv('../../day11_텍스트전처리_분석/data/reviews_band.csv')

all_tokens = []
for text in reviews['text']:
    cleaned = clean_text(text)
    words = [token.form for token in kiwi.tokenize(cleaned)
             if token.tag.startswith(('NN', 'VA', 'VV')) and len(token.form) > 1
             and token.form not in stopwords]
    all_tokens.extend(words)

counter_general = Counter(all_tokens)
print("전체 토큰 수:", len(all_tokens))
print("상위 20개:")
for word, count in counter_general.most_common(20):
    print(f"  {word}: {count}")

In [ ]:
# [자가채점]
assert len(all_tokens) == 17212
assert counter_general.most_common(3) == [('자세', 1156), ('착용', 940), ('어깨', 893)]
assert counter_general['교정'] == 577
print("✅ 문제8 통과!")

### 해설 — 문제 8
- **접근법**: 리뷰마다 나온 토큰 리스트를 `append` 가 아니라 **`extend`** 로 이어 붙여야 '리스트의 리스트'가 아닌 **평평한 단어 리스트**가 됩니다. `Counter` 는 그런 리스트를 받아 `{단어: 횟수}` 를 만들어 줍니다.
- **흔한 실수**: `clean_text` 를 건너뛰고 원문을 바로 토큰화하면 `ㅋㅋ`·특수문자가 섞여 형태소 분석이 흔들립니다. **정제 → 토큰화 → 불용어 제거** 순서를 지키세요.
- **관찰**: 상위 20개를 보면 `자세·착용·어깨·교정·허리`(제품과 직결된 의미 있는 단어) 사이에 `사용·구매·제품·느낌` 같은 **어느 제품 리뷰에나 나오는 말**이 섞여 있습니다. 다음 문제에서 이것들을 직접 걸러 냅니다.

## 9. 도메인 불용어 만들기 — 이 리뷰에서만 의미 없는 말 걸러 내기
**배경**: 문제 8 의 상위 20개를 다시 보세요. `사용`·`구매`·`제품`·`느낌` 은 **어떤 제품 리뷰에나 나오는 말**이라 '이 밴드가 어떤 제품인지'를 하나도 알려 주지 않습니다. 일반 불용어 목록엔 이런 단어가 없어요 — **도메인마다 다르기 때문**입니다. 그래서 실무에서는 **빈도 상위를 눈으로 보고, 그 도메인에서만 의미 없는 고빈도어를 직접 목록으로 만듭니다**. 이것이 **도메인 불용어**입니다.

> ⚠️ `자세`·`착용`·`어깨`·`교정`·`허리` 는 **이 제품의 핵심**을 말해 주는 단어입니다. 빈도가 높다고 지우면 안 됩니다. 지울 것은 **'어느 제품 리뷰에나 나오는 메타 표현'** 뿐입니다.

**요구사항**:
- 아래 10개 단어를 원소로 갖는 **집합** `domain_stopwords` 를 만드세요 — `제품`, `사용`, `구매`, `상품`, `주문`, `배송`, `쿠팡`, `리뷰`, `느낌`, `생각`
- 문제 8 의 `all_tokens` 에서 `domain_stopwords` 에 든 단어를 제거한 리스트를 `final_tokens` 변수에 담고, 그 빈도를 세어 `Counter` 를 `counter_final` 변수에 담으세요.
- 제거 **전**(`counter_general`)과 **후**(`counter_final`)의 상위 10개를 나란히 출력해 무엇이 바뀌었는지 확인하세요. 상위 10개 단어 목록을 `top10_after` 리스트에 담으세요(단어만, 횟수 제외).

**예시**
```
제거 전 top10  →  자세, 착용, 어깨, 교정, 허리, 밴드, 사용, 구매, 바르, 느낌
제거 후 top10  →  자세, 착용, 어깨, 교정, 허리, 밴드, 바르, 펴지, 사이즈, 불편
                 ('사용'·'구매'·'느낌' 이 빠지고 '펴지'·'사이즈'·'불편' 이 올라온다)
counter_final['자세']  →  1156   (핵심 단어의 횟수는 그대로)
```
<details><summary>힌트</summary>

```text
접근방법:
- 제품 리뷰라면 어디에나 나올 메타 표현을 집합으로 모은다(요구사항의 10개 단어).
- 전체 토큰에서 그 집합에 든 단어만 빼고 다시 빈도를 센다.

세부구현:
1. 중괄호로 도메인 불용어 집합을 만든다
2. all_tokens 를 순회하며 그 집합에 없는 단어만 final_tokens 에 모은다
3. Counter 에 final_tokens 를 넣어 counter_final 을 만든다
4. 제거 전·후의 상위 10개를 출력해 비교하고, 제거 후 상위 10개의 단어만 `top10_after` 변수에 담는다
```

</details>

In [ ]:
domain_stopwords = {'제품', '사용', '구매', '상품', '주문', '배송', '쿠팡', '리뷰', '느낌', '생각'}

final_tokens = [word for word in all_tokens if word not in domain_stopwords]
counter_final = Counter(final_tokens)
top10_after = [word for word, _ in counter_final.most_common(10)]

print("제거 전 top10:", [word for word, _ in counter_general.most_common(10)])
print("제거 후 top10:", top10_after)

In [ ]:
# [자가채점]
assert isinstance(domain_stopwords, set)
assert domain_stopwords == {'제품', '사용', '구매', '상품', '주문', '배송', '쿠팡', '리뷰', '느낌', '생각'}
assert len(top10_after) == 10
for word in ['제품', '사용', '구매', '느낌']:
    assert word not in top10_after
for word in ['자세', '착용', '어깨', '교정', '허리', '펴지', '사이즈', '불편']:
    assert word in top10_after
assert counter_final['자세'] == 1156
assert counter_final['펴지'] == 249
print("✅ 문제9 통과!")

### 해설 — 문제 9
- **접근법**: 도메인 불용어는 **자동으로 정해지지 않습니다**. '빈도 상위를 출력한다 → 사람이 읽고 판단한다 → 목록을 만든다 → 다시 적용한다' 는 반복이 실무의 핵심 작업입니다. 여기서 판단 기준은 **'이 단어가 제품의 특성을 말해 주는가?'** 였습니다. `구매`·`배송`은 아니고, `교정`·`펴지`는 맞습니다.
- **흔한 실수**: 빈도가 높다는 이유만으로 `자세`·`착용` 을 지우면, 정작 이 제품을 설명하는 단어가 사라져 워드클라우드가 텅 빕니다. **빈도가 아니라 '의미가 있는가'로** 판단하세요.
- **대안**: 일반 불용어와 도메인 불용어를 하나의 집합으로 합쳐(`stopwords | domain_stopwords`) 토큰화 단계에서 한 번에 거를 수도 있습니다. 결과는 같습니다.

## 10. 워드클라우드 — 리뷰를 한 장의 그림으로
**배경**: 빈도 표는 숫자라 한눈에 안 들어옵니다. **워드클라우드**는 자주 나온 단어를 크게 그려 줘서 리뷰 전체의 인상을 한 장으로 보여 줍니다.

**요구사항**:
- 문제 9 의 `counter_final` 을 사용합니다. `WordCloud` 는 `{단어: 횟수}` 형태의 **딕셔너리**를 받는 `generate_from_frequencies` 메서드로 그림을 만듭니다(상위 100개면 충분합니다).
- `WordCloud` 를 만들 때 **`font_path=FONT_PATH` 를 반드시 넣으세요** — 빼면 한글이 전부 □□□ 로 깨집니다. 크기는 `width=800, height=500`, 배경은 `background_color='white'`, `max_words=100` 으로 지정한 WordCloud 객체를 `wc` 변수에 담으세요.
- `wc.generate_from_frequencies(...)`로 만든 이미지를 `cloud` 변수에 담으세요. `fig, ax = plt.subplots(figsize=(10, 6))` 로 Figure와 Axes를 만든 뒤 `ax.imshow(cloud)`로 표시하고, `ax.axis('off')` 로 축을 끄고 `ax.set_title(...)`로 제목을 단 뒤 `plt.show()`로 보여 주세요.

**예시**: 아래 완성 그래프와 같은 모양(`자세`·`착용`·`어깨` 가 가장 크게 보이는 구름)이면 됩니다. (이 문제는 자가채점이 없습니다.)
<details><summary>힌트</summary>

```text
접근방법:
- 빈도 Counter 의 상위 항목을 딕셔너리로 바꿔 워드클라우드에 넘긴다.
- 한글 폰트 경로를 지정하지 않으면 글자가 깨지니 반드시 넣는다.
- 완성된 구름 이미지는 matplotlib 의 이미지 표시 함수로 그린다.

세부구현:
1. WordCloud 객체를 폰트 경로·크기·배경색·최대 단어 수와 함께 만들어 wc 변수에 담는다
2. Counter 의 상위 100개를 딕셔너리로 바꿔 generate_from_frequencies 에 넘기고 결과를 cloud 변수에 담는다
3. fig, ax = plt.subplots(...)로 Figure와 Axes를 만들고 ax.imshow로 구름을 그린다
4. 축을 끄고 제목을 단 뒤 화면에 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day11_텍스트전처리_분석/images/과제/lv1_q10_wordcloud.png" width="640">

In [ ]:
wc = WordCloud(font_path=FONT_PATH, width=800, height=500,
               background_color='white', max_words=100)
cloud = wc.generate_from_frequencies(dict(counter_final.most_common(100)))
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(cloud, interpolation='bilinear')
ax.axis('off')
ax.set_title('자세교정밴드 리뷰 워드클라우드')
plt.show()

### 해설 — 문제 10
- **접근법**: `generate_from_frequencies` 는 이미 세어 둔 빈도를 그대로 씁니다(원문을 넘기는 `generate` 와 다릅니다 — 한국어는 공백 기준으로 잘라선 안 되니 **빈도 딕셔너리**를 넘기는 쪽이 맞습니다).
- **흔한 실수**: `font_path` 를 빼면 한글이 전부 네모(□)로 나옵니다. 윈도우는 폰트 경로가 `C:/Windows/Fonts/malgun.ttf` 입니다.
- **읽는 법**: 글자 크기는 **빈도에 비례**할 뿐, 긍정·부정을 뜻하지 않습니다. `불편` 이 크게 보인다고 나쁜 제품이라는 뜻은 아니에요('불편하지 않다'는 문장에도 등장하니까요). 워드클라우드는 **무엇을 많이 이야기하는가**를 보는 도구입니다.

## 11. 빈도 막대그래프 — top10 을 정확히 비교하기
**배경**: 워드클라우드는 인상적이지만 **정확한 비교**엔 약합니다(글자 크기로 240과 260을 구분할 수 있나요?). 정확한 비교에는 **막대그래프**가 맞습니다. 같은 데이터를 두 방식으로 그려 차이를 느껴 봅시다.

**요구사항**:
- 문제 9 의 `counter_final` 에서 상위 10개 `(단어, 횟수)` 튜플을 `top10` 리스트 변수에 담으세요. 각 튜플의 단어는 `words` 리스트 변수에, 횟수는 `counts` 리스트 변수에 나누어 담으세요.
- `fig, ax = plt.subplots(figsize=(9, 5))` 로 Figure와 Axes를 만든 뒤 `sns.barplot(x=words, y=counts, ax=ax, errorbar=None)`로 x축은 단어, y축은 등장 횟수인 막대그래프를 그리세요.
- 제목·축 이름을 달고 `plt.show()` 로 보여 주세요.

**예시**: 아래 완성 그래프와 같은 모양(`자세`(1156)부터 `불편`(204)까지 내림차순 막대 10개)이면 됩니다. (이 문제는 자가채점이 없습니다.)
<details><summary>힌트</summary>

```text
접근방법:
- Counter 의 상위 10개는 (단어, 횟수) 튜플 목록이다. 이를 단어 목록과 횟수 목록으로 분리해 막대로 그린다.

세부구현:
1. counter_final.most_common(10) 결과를 top10 리스트 변수에 담는다
2. 리스트 컴프리헨션으로 words 문자열 리스트와 counts 정수 리스트를 각각 만든다
3. fig, ax = plt.subplots(...)로 Figure와 Axes를 만든다
4. sns.barplot에 단어 목록과 횟수 목록 및 ax를 넘기고, ax.set_title/set_xlabel/set_ylabel로 꾸민다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day11_텍스트전처리_분석/images/과제/lv1_q11_bar.png" width="640">

In [ ]:
top10 = counter_final.most_common(10)
words = [w for w, _ in top10]
counts = [c for _, c in top10]
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=words, y=counts, color='steelblue', ax=ax, errorbar=None)
ax.set_title('자세교정밴드 리뷰 단어 빈도 top10')
ax.set_xlabel('단어')
ax.set_ylabel('등장 횟수')
plt.show()

### 해설 — 문제 11
- **접근법**: `most_common(10)` 은 `[('자세', 1156), ...]` 형태의 튜플 리스트입니다. 컴프리헨션 두 번으로 단어 축과 값 축을 분리하면 그대로 `sns.barplot(x=words, y=counts, ax=ax, errorbar=None)`에 넘길 수 있습니다.
- **흔한 실수**: 그래프마다 `fig, ax = plt.subplots()`로 새 Figure와 Axes를 만들지 않으면 앞의 워드클라우드 위에 막대가 겹쳐 그려질 수 있습니다.
- **비교**: 같은 데이터를 워드클라우드(문제 10)와 막대(문제 11)로 그려 보면 역할이 분명해집니다 — **인상 전달은 워드클라우드, 정확한 비교는 막대그래프**입니다. 보고서에는 둘을 함께 싣는 경우가 많습니다.

## 12. 문자열에서 필요한 패턴 찾기 — `find`·`search`·`findall`
**배경**: 고객 문의 한 줄에는 주문 코드가 여러 개 섞일 수 있습니다. 정확한 글자의 위치, 첫 번째 패턴, 모든 패턴은 서로 다른 도구로 찾습니다.

**요구사항**:
- 제공된 문자열 `message`에서 정확한 글자 `주문번호`의 시작 위치를 문자열 메서드 `find`로 찾아 `literal_index`에 담으세요.
- 영문 대문자 2개-숫자 3개 패턴을 문자열 변수 `code_pattern`에 담으세요.
- `code_pattern`의 첫 일치를 `re.search`로 찾고, 실제 문자열을 `first_code`에 담으세요. 검색 결과가 없을 수도 있으므로 `None`을 확인한 뒤 `group()`을 사용하세요.
- 같은 패턴의 모든 일치값을 `re.findall`로 찾아 리스트 `all_codes`에 담으세요.

**예시**
```
literal_index → 13
first_code     → 'CS-102'
all_codes      → ['CS-102', 'OD-305']
```
<details><summary>힌트</summary>

```text
접근방법:
- 글자 그대로의 위치는 문자열 찾기 메서드, 패턴의 첫 한 건은 정규표현식 첫 검색, 전부는 목록 검색을 쓴다.

세부구현:
1. message 에서 '주문번호'의 시작 위치를 찾아 `literal_index` 변수에 담는다
2. 영문 대문자 2개-숫자 3개를 뜻하는 패턴을 만든다
3. 첫 검색 결과가 있는지 확인하고 실제 일치 문자열을 `first_code` 변수에 담는다
4. 같은 패턴의 모든 일치값을 `all_codes` 변수에 담아 출력한다
```

</details>

In [ ]:
# [제공 코드] 패턴을 찾을 고객 문의입니다.
message = '문의번호 CS-102, 주문번호 OD-305, 연락처 010-1234-5678'
print(message)

In [ ]:
literal_index = message.find('주문번호')
code_pattern = r'[A-Z]{2}-\d{3}'
first_match = re.search(code_pattern, message)
first_code = first_match.group() if first_match is not None else None
all_codes = re.findall(code_pattern, message)

print('주문번호 위치:', literal_index)
print('첫 코드:', first_code)
print('모든 코드:', all_codes)

In [ ]:
# [자가채점]
assert literal_index == 13
assert code_pattern == r'[A-Z]{2}-\d{3}'
assert first_code == 'CS-102'
assert all_codes == ['CS-102', 'OD-305']
print("✅ 문제12 통과!")

### 해설 — 문제 12
- **접근법**: `str.find`는 정확한 글자의 위치, `re.search`는 패턴의 첫 일치, `re.findall`은 모든 일치값을 맡습니다. 같은 '찾기'라도 원하는 결과에 따라 함수가 달라집니다.
- **흔한 실수**: `re.search`는 문자열이 아니라 `Match` 객체를 돌려줍니다. 실제 코드 문자열은 `.group()`으로 꺼내며, 검색 실패 시 `None`이므로 먼저 확인해야 합니다.
- **주의**: 존재 여부만 볼 때 `if message.find(...)`는 쓰지 않습니다. 시작 위치 0은 거짓, 실패 값 -1은 참처럼 평가될 수 있으므로 `'글자' in message`가 더 안전합니다.